# heatmap for one specific file

In [2]:
from datetime import datetime
import pandas as pd
from pathlib import Path
import pickle as pl
import numpy as np
import seaborn as sns  
import matplotlib.pyplot as plt
import json
from datasets import load_dataset

/home/aditya/miniconda3/envs/myenv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
align_dir ='./results_align_matrix_ag_news/ag_news_linear_[0.01, 0.4, 0.3, 0.29]'
datainfo_dir = './results_datainfo_ag_news/ag_news_(0.01, 0.4, 0.3, 0.29)'

## my input features
alignment_matrix = np.load(f"{align_dir}/alignment_matrix_linear.npy")

with open(f'{datainfo_dir}/accuracy_arr.pkl','rb') as f:
    accuracy_arr = pl.load(f)
with open(f'{datainfo_dir}/dataset_info.json') as f:
    dataset_info =json.load(f)


def compute_per_class_alignment_heatmap(alignment_matrix, datainfo_dir, dataset_name='ag_news'):
    """Compute mean alignment score per class per pseudo-expert.
    
    alignment_matrix shape: (N, K)
        N = number of samples
        K = number of pseudo-experts
    """
    
    # Load indices
    with open(f'{datainfo_dir}/dataset_info.json') as f:
        valid_indices = json.load(f)['indices_D']
    
    # Load TRAIN dataset
    if dataset_name == 'ag_news':
        dataset = load_dataset('ag_news', split='train')
    elif dataset_name == 'snli':
        dataset = load_dataset('snli', split='train')
    else:  # yelp
        dataset = load_dataset('yelp_review_full', split='train')
    
    # Get class for each index
    labels = np.array([dataset[idx]['label'] for idx in valid_indices])
    num_classes = int(labels.max()) + 1

    N, K = alignment_matrix.shape
    if N != len(labels):
        raise ValueError(
            f"Mismatch: alignment_matrix has {N} samples, but labels has {len(labels)} entries."
        )
    
    # Compute mean alignment per class per pseudo-expert
    per_class_alignment = np.zeros((K, num_classes))
    
    for k in range(K):
        for c in range(num_classes):
            class_mask = labels == c
            # Mean alignment score for class c in pseudo-expert k
            per_class_alignment[k, c] = alignment_matrix[class_mask, k].mean()
    
    return per_class_alignment, num_classes


def plot_alignment_heatmap(per_class_alignment, dataset_name='ag_news', save_path=None):
    """Plot heatmap of mean alignment per class across pseudo-experts."""
    
    class_names_map = {
        'ag_news': ['World', 'Sports', 'Business', 'Sci/Tech'],
        'snli': ['Entailment', 'Neutral', 'Contradiction'],
        'yelp_review': ['1★', '2★', '3★', '4★', '5★'],
        'yelp_review_full': ['1★', '2★', '3★', '4★', '5★']
    }
    
    class_names = class_names_map.get(
        dataset_name,
        [f'C{i}' for i in range(per_class_alignment.shape[1])]
    )
    
    K = per_class_alignment.shape[0]
    
    plt.figure(figsize=(12, 8))
    
    sns.heatmap(
        per_class_alignment.T,
        annot=True,
        fmt='.3f',
        cmap='RdYlGn',
        xticklabels=[f'θ_{i}' for i in range(K)],
        yticklabels=class_names,
        cbar_kws={'label': 'Mean Alignment Score'},
        linewidths=0.5
    )
    
    plt.xlabel('Pseudo-Expert Index', fontsize=12)
    plt.ylabel('Class', fontsize=12)
    plt.title('Mean Alignment Score per Class across Pseudo-Experts\n(Training Data)', fontsize=14)
    plt.tight_layout()
    
    # if save_path:
    #     plt.savefig(save_path, dpi=300, bbox_inches='tight')
    #     print(f"✅ Saved: {save_path}")
    #     plt.close()
    # else:
    plt.show()


# Run
per_class_alignment, num_classes = compute_per_class_alignment_heatmap(
    alignment_matrix, datainfo_dir, 'ag_news'
)

plot_alignment_heatmap(
    per_class_alignment,
    'ag_news',
    f"{align_dir}/per_class_alignment_heatmap.png"
)

# Print stats
print("\nPer-Class Alignment across Pseudo-Experts:")
print("=" * 60)
class_names = ['World', 'Sports', 'Business', 'Sci/Tech']
for i, name in enumerate(class_names):
    print(f"{name:12s}: {per_class_alignment[:, i]}")
print("=" * 60)

In [ ]:
# sns.heatmap(alignment_matrix)
# plt.show()

# heatmaps for all the files

In [11]:
from pathlib import Path
import json
import re
import numpy as np
from datasets import load_dataset
import matplotlib.pyplot as plt
import seaborn as sns


# =========================================================
# DATASET LOADING
# =========================================================
def load_train_dataset(dataset_name: str):
    if dataset_name == "ag_news":
        return load_dataset("ag_news", split="train")
    if dataset_name == "snli":
        return load_dataset("snli", split="train")
    if dataset_name in {"yelp_review", "yelp_review_full"}:
        return load_dataset("yelp_review_full", split="train")
    raise ValueError(f"Unsupported dataset_name: {dataset_name}")


def get_class_names(dataset_name: str, num_classes: int):
    class_names_map = {
        "ag_news": ["World", "Sports", "Business", "Sci/Tech"],
        "snli": ["Entailment", "Neutral", "Contradiction"],
        "yelp_review": ["1★", "2★", "3★", "4★", "5★"],
        "yelp_review_full": ["1★", "2★", "3★", "4★", "5★"],
    }
    return class_names_map.get(dataset_name, [f"C{i}" for i in range(num_classes)])


# =========================================================
# PARSING
# =========================================================
def extract_proportion_tuple(name: str):
    m = re.search(r"(\[[^\]]+\]|\([^)]*\))", name)
    if not m:
        return None
    raw = m.group(1)[1:-1]
    return tuple(float(x.strip()) for x in raw.split(",") if x.strip())


def parse_align_dir_name(name: str, dataset_name: str):
    """
    Example:
      ag_news_linear_[0.01, 0.4, 0.3, 0.29]
      ag_news_model_baseline_[0.01, 0.4, 0.3, 0.29]

    Returns:
      {
        "dataset": "ag_news",
        "method": "linear" or "model_baseline",
        "proportion": (...)
      }
    """
    prop = extract_proportion_tuple(name)
    if prop is None:
        return None

    # Remove the proportion part
    prefix = re.sub(r"(\[[^\]]+\]|\([^)]*\))", "", name).rstrip("_")

    expected_prefix = f"{dataset_name}_"
    if not prefix.startswith(expected_prefix):
        return None

    method = prefix[len(expected_prefix):]
    if not method:
        return None

    return {
        "dataset": dataset_name,
        "method": method,
        "proportion": prop,
    }


def parse_datainfo_dir_name(name: str):
    """
    Example:
      ag_news_(0.01, 0.4, 0.3, 0.29)

    Returns:
      {
        "dataset": "ag_news",
        "proportion": (0.01, 0.4, 0.3, 0.29)
      }
    """
    prop = extract_proportion_tuple(name)
    if prop is None:
        return None

    prefix = name.split("(")[0].rstrip("_")
    return {
        "dataset": prefix,
        "proportion": prop,
    }


# =========================================================
# MATCHING
# =========================================================
def collect_alignment_dirs(align_root: Path, dataset_name: str, method: str | None = None):
    out = {}
    for p in align_root.iterdir():
        if not p.is_dir():
            continue
        parsed = parse_align_dir_name(p.name, dataset_name)
        if parsed is None:
            continue
        if method is not None and parsed["method"] != method:
            continue
        out[parsed["proportion"]] = p
    return out


def collect_datainfo_dirs(datainfo_root: Path, dataset_name: str):
    out = {}
    for p in datainfo_root.iterdir():
        if not p.is_dir():
            continue
        parsed = parse_datainfo_dir_name(p.name)
        if parsed is None:
            continue
        if parsed["dataset"] != dataset_name:
            continue
        out[parsed["proportion"]] = p
    return out


def get_matched_pairs(
    align_root,
    datainfo_root,
    dataset_name="ag_news",
    method="linear",
    selected_proportions=None,
    max_pairs=None,
):
    align_root = Path(align_root)
    datainfo_root = Path(datainfo_root)

    align_dirs = collect_alignment_dirs(align_root, dataset_name, method=method)
    datainfo_dirs = collect_datainfo_dirs(datainfo_root, dataset_name)

    common_props = sorted(set(align_dirs) & set(datainfo_dirs))

    if selected_proportions is not None:
        selected_set = {tuple(map(float, p)) for p in selected_proportions}
        common_props = [p for p in common_props if p in selected_set]

    if max_pairs is not None:
        common_props = common_props[:max_pairs]

    return [(prop, align_dirs[prop], datainfo_dirs[prop]) for prop in common_props]


# =========================================================
# CORE COMPUTATION
# =========================================================
def compute_per_class_alignment(alignment_matrix: np.ndarray, labels: np.ndarray) -> np.ndarray:
    """
    alignment_matrix shape: (N, K)
      N = samples
      K = pseudo-experts

    Returns:
      per_class_alignment shape: (K, C)
    """
    if alignment_matrix.ndim != 2:
        raise ValueError(f"Expected 2D alignment_matrix, got shape {alignment_matrix.shape}")

    N, K = alignment_matrix.shape
    if N != len(labels):
        raise ValueError(
            f"Mismatch: alignment_matrix has {N} samples, but labels has {len(labels)} entries."
        )

    num_classes = int(labels.max()) + 1
    out = np.full((K, num_classes), np.nan, dtype=np.float32)

    for k in range(K):
        for c in range(num_classes):
            mask = labels == c
            if np.any(mask):
                out[k, c] = alignment_matrix[mask, k].mean()

    return out


def get_labels_from_datainfo(dataset, datainfo_json_path: Path):
    with open(datainfo_json_path, "r") as f:
        dataset_info = json.load(f)

    valid_indices = dataset_info["indices_D"]
    labels = np.array([dataset[idx]["label"] for idx in valid_indices], dtype=np.int64)
    return labels, dataset_info


# =========================================================
# FILE DISCOVERY
# =========================================================
def find_alignment_matrix_file(align_dir: Path, method="linear"):
    preferred = align_dir / f"alignment_matrix_{method}.npy"
    if preferred.exists():
        return preferred

    generic = sorted(align_dir.glob("alignment_matrix*.npy"))
    if generic:
        if len(generic) > 1:
            print(f"Warning: multiple alignment files in {align_dir}, using {generic[0].name}")
        return generic[0]

    raise FileNotFoundError(f"No alignment matrix file found in {align_dir}")


# =========================================================
# HEATMAP
# =========================================================
def save_alignment_heatmap(
    per_class_alignment: np.ndarray,
    class_proportions: np.ndarray,
    dataset_name: str,
    save_path: Path,
    title: str | None = None,
):
    num_classes = per_class_alignment.shape[1]
    K = per_class_alignment.shape[0]

    class_names = get_class_names(dataset_name, num_classes)
    yticklabels = [
        f"{class_names[i]} ({100 * class_proportions[i]:.1f}%)"
        for i in range(num_classes)
    ]

    plt.figure(figsize=(12, 8))
    sns.heatmap(
        per_class_alignment.T,
        annot=True,
        fmt=".3f",
        cmap="RdYlGn",
        xticklabels=[f"θ_{i}" for i in range(K)],
        yticklabels=yticklabels,
        cbar_kws={"label": "Mean Alignment Score"},
        linewidths=0.5,
    )

    plt.xlabel("Pseudo-Expert Index", fontsize=12)
    plt.ylabel("Class", fontsize=12)
    plt.title(
        title or "Mean Alignment Score per Class across Pseudo-Experts",
        fontsize=14,
    )
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches="tight", format="png")
    plt.close()


# =========================================================
# GENERATE ONE
# =========================================================
def generate_one_per_class_alignment(
    align_dir: Path,
    datainfo_dir: Path,
    dataset,
    output_root: Path,
    dataset_name="ag_news",
    method="linear",
    datainfo_filename="dataset_info.json",
    output_filename="per_class_alignment.npy",
    metadata_filename="metadata.json",
    heatmap_filename="per_class_alignment_heatmap.png",
):
    datainfo_path = datainfo_dir / datainfo_filename
    if not datainfo_path.exists():
        raise FileNotFoundError(f"Missing datainfo file: {datainfo_path}")

    alignment_path = find_alignment_matrix_file(align_dir, method=method)
    alignment_matrix = np.load(alignment_path)

    labels, dataset_info = get_labels_from_datainfo(dataset, datainfo_path)
    per_class_alignment = compute_per_class_alignment(alignment_matrix, labels)

    num_classes = per_class_alignment.shape[1]
    class_counts = np.bincount(labels, minlength=num_classes)
    class_proportions = class_counts / class_counts.sum()

    out_dir = output_root / datainfo_dir.name
    out_dir.mkdir(parents=True, exist_ok=True)

    np.save(out_dir / output_filename, per_class_alignment)

    save_alignment_heatmap(
        per_class_alignment=per_class_alignment,
        class_proportions=class_proportions,
        dataset_name=dataset_name,
        save_path=out_dir / heatmap_filename,
        title=f"Per-Class Alignment Heatmap\n{datainfo_dir.name}",
    )

    metadata = {
        "dataset_name": dataset_name,
        "method": method,
        "proportion_key": list(extract_proportion_tuple(datainfo_dir.name)),
        "alignment_dir": str(align_dir),
        "alignment_file_used": str(alignment_path),
        "datainfo_dir": str(datainfo_dir),
        "alignment_shape": list(alignment_matrix.shape),
        "per_class_alignment_shape": list(per_class_alignment.shape),
        "class_counts": class_counts.tolist(),
        "class_proportions": class_proportions.tolist(),
        "indices_count": int(len(dataset_info["indices_D"])),
        "saved_files": {
            "matrix": output_filename,
            "heatmap": heatmap_filename,
            "metadata": metadata_filename,
        },
    }

    with open(out_dir / metadata_filename, "w") as f:
        json.dump(metadata, f, indent=2)

    return {
        "proportion_key": extract_proportion_tuple(datainfo_dir.name),
        "align_dir": align_dir,
        "datainfo_dir": datainfo_dir,
        "alignment_file": alignment_path,
        "output_dir": out_dir,
        "heatmap_path": out_dir / heatmap_filename,
    }


# =========================================================
# DRIVER
# =========================================================
def generate_per_class_alignments(
    align_root,
    datainfo_root,
    output_root,
    dataset_name="ag_news",
    method="linear",
    selected_proportions=None,
    max_pairs=None,
):
    align_root = Path(align_root)
    datainfo_root = Path(datainfo_root)
    output_root = Path(output_root)
    output_root.mkdir(parents=True, exist_ok=True)

    dataset = load_train_dataset(dataset_name)

    matched_pairs = get_matched_pairs(
        align_root=align_root,
        datainfo_root=datainfo_root,
        dataset_name=dataset_name,
        method=method,
        selected_proportions=selected_proportions,
        max_pairs=max_pairs,
    )

    if not matched_pairs:
        raise RuntimeError(
            f"No matching proportion directories found for dataset={dataset_name}, method={method}"
        )

    print(f"Found {len(matched_pairs)} matched pairs for method={method}.\n")

    results = []
    for proportion_key, align_dir, datainfo_dir in matched_pairs:
        print(f"Processing proportion: {proportion_key}")
        result = generate_one_per_class_alignment(
            align_dir=align_dir,
            datainfo_dir=datainfo_dir,
            dataset=dataset,
            output_root=output_root,
            dataset_name=dataset_name,
            method=method,
        )
        results.append(result)
        print(f"  alignment file: {result['alignment_file'].name}")
        print(f"  saved matrix  : {result['output_dir'] / 'per_class_alignment.npy'}")
        print(f"  saved heatmap : {result['heatmap_path']}\n")

    return results

In [12]:
method = "model_baseline"

results = generate_per_class_alignments(
    align_root="./results_align_matrix_ag_news",
    datainfo_root="./results_datainfo_ag_news",
    method= method,
    output_root= f"./results_per_class_alignment_ag_news_{method}",
    dataset_name="ag_news",
    max_pairs=30
)

Found 30 matched pairs for method=model_baseline.

Processing proportion: (0.0, 0.45, 0.09, 0.46)
  alignment file: alignment_matrix_model_baseline.npy
  saved matrix  : results_per_class_alignment_ag_news_model_baseline/ag_news_(0.0, 0.45, 0.09, 0.46)/per_class_alignment.npy
  saved heatmap : results_per_class_alignment_ag_news_model_baseline/ag_news_(0.0, 0.45, 0.09, 0.46)/per_class_alignment_heatmap.png

Processing proportion: (0.01, 0.4, 0.3, 0.29)
  alignment file: alignment_matrix_model_baseline.npy
  saved matrix  : results_per_class_alignment_ag_news_model_baseline/ag_news_(0.01, 0.4, 0.3, 0.29)/per_class_alignment.npy
  saved heatmap : results_per_class_alignment_ag_news_model_baseline/ag_news_(0.01, 0.4, 0.3, 0.29)/per_class_alignment_heatmap.png

Processing proportion: (0.02, 0.34, 0.37, 0.27)
  alignment file: alignment_matrix_model_baseline.npy
  saved matrix  : results_per_class_alignment_ag_news_model_baseline/ag_news_(0.02, 0.34, 0.37, 0.27)/per_class_alignment.npy
  sa